In [ ]:
knitr::opts_chunk$set(echo = FALSE,message = FALSE,warning = FALSE)
options(scipen=999)


In [ ]:
if (!exists("book_load_packages", mode = "function")) source("scripts/book_runtime.R")
book_load_packages(fredr,
               dplyr,
               tidyr,
               lubridate,
               caret,
               partykit,
               pROC,
               devtools,
               httr,
               DT,
               httr,
               visNetwork,
               dkstat,
               data.table,
               pROC,
               gridExtra,
               glue,
               igraph,
               visNetwork,
               rPref,
               progress,
               quantmod,
               rvest,
               ggrepel)
source(file = "money_theme.R")


# Economic Release Calendar


In [ ]:
if (file.exists("Renviron.site")) readRenviron(path = "Renviron.site")
if (.book_runtime$mode != "replay") fredr::fredr_set_key(key = Sys.getenv("FRED_API"))


In [ ]:
df_series = do.call("rbind",lapply(c(0,1000*seq(10)),function(offset_) {
book_fred_metadata("fredr_tags_series", "nsa",order_by = "popularity",sort_order = "desc",offset = offset_)
})) %>%
  filter(frequency_short %in% c("M","W","Q")) %>%
  dplyr::select(id,title,frequency_short,last_updated,popularity)


## Function


In [ ]:
draw_plots = function(data) {

# All time plot

series_id = unique(data$series_id)
description = unique(data$title)
p1 = data %>%
  ggplot(.,aes(x=date,y=value)) +
  geom_line() +
  theme_money_printer_go_brrr(12) +
  scale_x_date(date_labels = "%Y",date_breaks = "5 years") +
  labs(x=NULL,y=NULL,title=glue("{description}",subtitle="{series_id}"))

# Last year
p2 = data %>%
  filter(date>=book_date()-years(1)) %>%
  ggplot(.,aes(x=date,y=value)) +
  geom_line() +
  theme_money_printer_go_brrr(12) +
  scale_x_date(date_labels = "%b",date_breaks = "1 month") +
  geom_vline(xintercept = book_date(),linetype=2) +
  labs(x=NULL,y=NULL)

# SMA
p3 = data %>%
  arrange(date) %>%
  dplyr::mutate(sma200=TTR::SMA(value,n=200)) %>%
  dplyr::mutate(value=value-sma200) %>%
  filter(date>=book_date()-years(1)) %>%
  ggplot(.,aes(x=date,y=value)) +
  geom_line() +
  theme_money_printer_go_brrr(12) +
  scale_x_date(date_labels = "%b",date_breaks = "1 month") +
  geom_hline(yintercept = 0,linetype=2) +
  labs(x=NULL,y=NULL)

# RSI
p4 = data %>%
  arrange(date) %>%
  dplyr::mutate(rsi=TTR::RSI(value,n=30)) %>%
  filter(date>=book_date()-years(1)) %>%
  ggplot(.,aes(x=date,y=rsi)) +
  geom_line() +
  theme_money_printer_go_brrr(12) +
  scale_x_date(date_labels = "%b",date_breaks = "1 month") +
  geom_hline(yintercept = c(20,70),linetype=2) +
  labs(x=NULL,y=NULL)

p5 = grid.arrange(p1,p2,p3,p4,ncol=1)
print(p5)

}


## All


In [ ]:
df_series %>%
  head(100) %>%
  DT::datatable()


In [ ]:
top_x = df_series %>%
  head(25) %>%
  pull(id)


In [ ]:
df_data = rbindlist(lapply(top_x,function(series_id) {

  series_info = book_fred_metadata("fredr_series", series_id) %>%
    dplyr::select(id,title,frequency,units_short,seasonal_adjustment_short,popularity)
  fredr_cached(series_id = series_id) %>%
    tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=book_date(),by="1 day")) %>%
    tidyr::fill(value,.direction = "down") %>%
    tidyr::fill(series_id,.direction = "down") %>%
      dplyr::select(-realtime_start,-realtime_end) %>%
      tidyr::crossing(series_info)

}),fill=TRUE)


In [ ]:
for (series in unique(df_data$series_id)) {
  draw_plots(data = df_data %>% filter(series_id==series))
}


## Last week


In [ ]:
df_series %>%
  filter(last_updated>=book_date()-days(7)) %>%
  DT::datatable()


In [ ]:
top_x = df_series %>%
  filter(last_updated>=book_date()-days(7)) %>%
  head(25) %>%
  pull(id)


In [ ]:
df_data = rbindlist(lapply(top_x,function(series_id) {

  series_info = book_fred_metadata("fredr_series", series_id) %>%
    dplyr::select(id,title,frequency,units_short,seasonal_adjustment_short,popularity)
  fredr_cached(series_id = series_id) %>%
    tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=book_date(),by="1 day")) %>%
    tidyr::fill(value,.direction = "down") %>%
    tidyr::fill(series_id,.direction = "down") %>%
      dplyr::select(-realtime_start,-realtime_end) %>%
      tidyr::crossing(series_info)

}),fill=TRUE)


In [ ]:
for (series in unique(df_data$series_id)) {
  draw_plots(data = df_data %>% filter(series_id==series))
}


## Unemployment


In [ ]:
top_x = c("ICSA","UNRATE","CCSA","PAYEMS","CIVPART","HOHWMN02USM065S","CES0500000017","ADPWNUSNERSA","M08343USM232SNBR","LES1252881600Q")
df_data = rbindlist(lapply(top_x,function(series_id) {

  series_info = book_fred_metadata("fredr_series", series_id) %>%
    dplyr::select(id,title,frequency,units_short,seasonal_adjustment_short,popularity)
  fredr_cached(series_id = series_id) %>%
    tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=book_date(),by="1 day")) %>%
    tidyr::fill(value,.direction = "down") %>%
    tidyr::fill(series_id,.direction = "down") %>%
      dplyr::select(-realtime_start,-realtime_end) %>%
      tidyr::crossing(series_info)

}),fill=TRUE)


In [ ]:
for (series in unique(df_data$series_id)) {
  draw_plots(data = df_data %>% filter(series_id==series))
}


## U6 vs UNRATE


In [ ]:
unrates = c("UNRATE","U6RATE")
df_u6 = rbindlist(lapply(unrates,function(series_id) {

  series_info = book_fred_metadata("fredr_series", series_id) %>%
    dplyr::select(id,title,frequency,units_short,seasonal_adjustment_short,popularity)
  fredr_cached(series_id = series_id) %>%
    tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=book_date(),by="1 day")) %>%
    tidyr::fill(value,.direction = "down") %>%
    tidyr::fill(series_id,.direction = "down") %>%
      dplyr::select(-realtime_start,-realtime_end) %>%
      tidyr::crossing(series_info)

}),fill=TRUE)

df_u6 %>%
  filter(date>=ymd("1994-01-01")) %>%
  ggplot(aes(x=date,y=value,color=series_id)) +
   geom_line()


df_u6 %>%
  distinct(series_id,date,value) %>%
  spread(series_id,value) %>%
  arrange(desc(date)) %>%
  dplyr::mutate(diff=U6RATE-UNRATE) %>%
  filter(date>=ymd("1994-01-01")) %>%
  ggplot(aes(x=date,y=diff)) +
   geom_line()
